In [10]:
## Loading the training dataset and check the data

import numpy as np
from pathlib import Path

d = np.load(Path("/Users/azza/Desktop/AI agents/ml/datasets/weather_agent_random_train.fixed.npz"))
X, y = d["dataset"], d["label"]

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)
print("classes:", np.unique(y), "counts:", np.bincount(y.astype(int)))
print("sample[0] shape:", X[0].shape)
print("sample[0] first 10:", X[0].reshape(-1)[:10])
labels = np.unique(y)
print("Unique labels:", labels)

X: (720, 2400) float32
y: (720,) int64
classes: [0 1 2] counts: [240 240 240]
sample[0] shape: (2400,)
sample[0] first 10: [12.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
Unique labels: [0 1 2]


In [11]:
## Random Forest code
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from joblib import dump

# ---- paths ----
base = Path("/Users/azza/Desktop/AI agents/ml/datasets")
data_path = base / "weather_agent_random_train.fixed.npz"
model_path = base / "weather_agent_random_train_rf.joblib"

# ---- load ----
d = np.load(data_path)
X = d["dataset"]
y = d["label"]

print("Loaded:", data_path)
print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)

# ---- flatten for RF: [N, ...] -> [N, D] ----
X2 = X.reshape(X.shape[0], -1)
y = y.astype(int)

print("Flattened X2:", X2.shape)

# ---- train/test split ----
X_train, X_test, y_train, y_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y
)

# ---- model ----
rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

# ---- train ----
rf.fit(X_train, y_train)

# ---- evaluate ----
pred = rf.predict(X_test)
acc = accuracy_score(y_test, pred)
print("\nAccuracy:", acc)

print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))
print("\nClassification report:\n", classification_report(y_test, pred))

# ---- save ----
dump(rf, model_path)
print("\nSaved RF model to:", model_path)


Loaded: /Users/azza/Desktop/AI agents/ml/datasets/weather_agent_random_train.fixed.npz
X: (720, 2400) float32
y: (720,) int64
Flattened X2: (720, 2400)

Accuracy: 0.9930555555555556

Confusion matrix:
 [[47  1  0]
 [ 0 48  0]
 [ 0  0 48]]

Classification report:
               precision    recall  f1-score   support

           0       1.00      0.98      0.99        48
           1       0.98      1.00      0.99        48
           2       1.00      1.00      1.00        48

    accuracy                           0.99       144
   macro avg       0.99      0.99      0.99       144
weighted avg       0.99      0.99      0.99       144


Saved RF model to: /Users/azza/Desktop/AI agents/ml/datasets/weather_agent_random_train_rf.joblib


In [12]:
#Loading testig dataset and evaluate the model on it
import numpy as np
from joblib import load
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report

rf = load(Path("/Users/azza/Desktop/AI agents/ml/datasets/weather_agent_random_train_rf.joblib"))

d = np.load(Path("/Users/azza/Desktop/AI agents/ml/datasets/weather_agent_test.fixed.npz"))
X_test = d["dataset"]
y_test = d["label"] if "label" in d.files else None

print("X_test:", X_test.shape)
print("Model expects:", rf.n_features_in_)

# Ensure feature match
if X_test.ndim != 2 or X_test.shape[1] != rf.n_features_in_:
    raise ValueError(f"Feature mismatch: got {X_test.shape}, expected (*, {rf.n_features_in_})")

y_pred = rf.predict(X_test)

if y_test is not None:
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
else:
    print("Predictions (first 20):", y_pred[:20])

X_test: (106, 2400)
Model expects: 2400
Accuracy: 0.9622641509433962
              precision    recall  f1-score   support

           0       0.90      1.00      0.95        37
           1       1.00      0.87      0.93        30
           2       1.00      1.00      1.00        39

    accuracy                           0.96       106
   macro avg       0.97      0.96      0.96       106
weighted avg       0.97      0.96      0.96       106

